# YouTube Video Brand Name Extraction Tool

Uses the OpenAI API to automatically identify real-world brand names from the **title** and **description** of YouTube videos.

Output format: one entry per brand, including a `summary` object.

See `README.md` for setup instructions.

## Step 1: Install and Import Packages

In [39]:
import json
import time
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

print("Packages loaded")

Packages loaded


## Step 2: Configure Parameters

In [40]:
# ============================================================
# OpenAI API Key
# ============================================================
OPENAI_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your own API key

# ============================================================
# Input JSON Path
# ============================================================
JSON_FILE_PATH = "YOUR_INPUT_FILE.json"  # Replace with your JSON file path

# ============================================================
# JSON Field Names (mapped to your data structure)
# ============================================================
TITLE_FIELD       = "title"    # Video title field
DESCRIPTION_FIELD = "description"  # Video description field (set to "" if none)
VIDEO_ID_FIELD    = "video_id"     # Video ID field
CHANNEL_NAME_FIELD = "channel"     # Channel name field
CHANNEL_ID_FIELD   = "channel_id"   # Channel ID field

# ============================================================
# Model Settings
# ============================================================
MODEL      = "gpt-5.4-mini"  # gpt-4o-mini recommended (cost-effective); use gpt-4o for higher accuracy
BATCH_SIZE = 10              # Number of items per batch (5~20 recommended)
SLEEP_SEC  = 1               # Seconds to wait between batches

# ============================================================
# Output JSON Path
# ============================================================
OUTPUT_JSON = "YOUR_OUTPUT_FILE.json"

print("Parameters configured")

Parameters configured


## Step 3: Load Data

In [ ]:
with open(JSON_FILE_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Supports either a list format or a {"videos": [...]} wrapped format
if isinstance(raw_data, list):
    videos = raw_data
elif isinstance(raw_data, dict):
    for key, val in raw_data.items():
        if isinstance(val, list):
            videos = val
            print(f"ℹ Detected data under key:'{key}'")
            break
    else:
        videos = [raw_data]

df = pd.DataFrame(videos)
print(f"Loaded {len(df)} videos")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# Fill missing values
for field in [TITLE_FIELD, DESCRIPTION_FIELD, VIDEO_ID_FIELD, CHANNEL_NAME_FIELD, CHANNEL_ID_FIELD]:
    if field and field in df.columns:
        df[field] = df[field].fillna("")

print("Field processing complete")

## Step 4: Define the Brand Extraction Function

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """You are a professional brand identification AI.
Your task is to do two things based on the title and description of a YouTube video:
(1) Determine whether the video content is in Traditional Chinese
(2) Identify real-world, actually existing brand names

Language detection rules:
- Set is_traditional_chinese to true: the title/description is mainly written in Traditional Chinese
- Set is_traditional_chinese to false: English, Japanese, Simplified Chinese, or other languages
- If the title only contains hashtags, symbols, or too little text to judge, set it to false

Brand extraction rules:
1. Only extract "real-world, actually existing commercial brands", e.g., Nike, Apple, Starbucks, IKEA, HotToys, etc.
2. Do NOT include: personal names, place names, generic terms, or show names (unless the show itself is a brand)
3. Use the "standardized name" for brands (e.g., hot toys → HotToys, 蘋果 → Apple)
4. If the same brand is mentioned multiple times within the title or description of the SAME video, list it only once; however, each video is independent, so the same brand may appear separately across multiple videos
5. If no brand is found, return an empty list [] for brands
6. Return only JSON format, with no other text"""


def build_user_prompt(batch: list[dict]) -> str:
    lines = []
    for i, item in enumerate(batch):
        title = item.get("title", "")
        desc  = item.get("description", "")
        if len(desc) > 500:
            desc = desc[:500] + "..."
        lines.append(f"[{i+1}]\nTitle: {title}\nDescription: {desc}")

    prompt  = "Below are multiple YouTube videos. For each one: (1) determine whether it is in Traditional Chinese (2) list the brand names (standardized).\n\n"
    prompt += "\n\n".join(lines)
    prompt += "\n\nPlease return the result in the following JSON format:\n"
    prompt += json.dumps(
        [
            {"index": 1, "is_traditional_chinese": True,  "brands": ["HotToys", "Marvel"]},
            {"index": 2, "is_traditional_chinese": False, "brands": []}
        ],
        ensure_ascii=False
    )
    return prompt


def extract_brands_batch(batch: list[dict]) -> list[dict]:
    """Returns {is_traditional_chinese, brands} for each video in the batch"""
    user_prompt = build_user_prompt(batch)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    content = response.choices[0].message.content
    parsed  = json.loads(content)

    results = []
    if isinstance(parsed, list):
        results = parsed
    elif isinstance(parsed, dict):
        for val in parsed.values():
            if isinstance(val, list):
                results = val
                break

    output = []
    for item in results:
        if isinstance(item, dict):
            output.append({
                "is_traditional_chinese": item.get("is_traditional_chinese", False),
                "brands": item.get("brands", [])
            })
        else:
            output.append({"is_traditional_chinese": False, "brands": []})

    while len(output) < len(batch):
        output.append({"is_traditional_chinese": False, "brands": []})

    return output[:len(batch)]


print("Function defined")


## Step 5: Run Brand Extraction in Batches

In [ ]:
# Prepare input data (keep the original fields)
records = []
for _, row in df.iterrows():
    original_record = row.to_dict()
    records.append({
        "original": original_record,  # Keep the full original record
        "title": str(original_record.get(TITLE_FIELD, "")),
        "description": str(original_record.get(DESCRIPTION_FIELD, "")) if DESCRIPTION_FIELD and DESCRIPTION_FIELD in df.columns else "",
    })

# Call the API in batches
all_results = []  # list of (record, {is_traditional_chinese, brands})
total_batches = (len(records) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"{len(records)} records total, split into {total_batches} batches ({BATCH_SIZE} per batch) | Model: {MODEL}\n")

for i in tqdm(range(0, len(records), BATCH_SIZE), desc="Brand extraction progress"):
    batch = records[i : i + BATCH_SIZE]
    try:
        results_batch = extract_brands_batch(batch)
    except Exception as e:
        print(f"\n Error in batch {i//BATCH_SIZE+1}: {e}")
        results_batch = [{"is_traditional_chinese": False, "brands": []} for _ in batch]

    for rec, res in zip(batch, results_batch):
        all_results.append((rec, res))

    if i + BATCH_SIZE < len(records):
        time.sleep(SLEEP_SEC)

print(f"\n Done! Processed {len(all_results)} videos")

## Step 6: Format into the Target Output & Preview

In [ ]:
# Only keep videos that are: Traditional Chinese + have brands
output_records = []
removed_not_tc = 0
removed_no_brand = 0

for rec, res in all_results:
    is_tc = res.get("is_traditional_chinese", False)
    brands = res.get("brands", [])

    # Drop non-Traditional-Chinese videos
    if not is_tc:
        removed_not_tc += 1
        continue

    # Drop videos with no brands
    if not brands:
        removed_no_brand += 1
        continue

    # Copy the original data to avoid mutating the original object
    output_item = dict(rec.get("original", {}))

    # Add the brand field
    output_item["brand名稱_標準化"] = brands

    output_records.append(output_item)

print(f"Stats:")
print(f"  • Original video count: {len(all_results)}")
print(f"  • Removed (not Traditional Chinese): {removed_not_tc}")
print(f"  • Removed (no brand): {removed_no_brand}")
print(f"  • Final output count: {len(output_records)}")
print(f"\n Preview of first 3 output records:")
for item in output_records[:3]:
    print(json.dumps(item, ensure_ascii=False, indent=2))

## Step 7: Save Output JSON

In [ ]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output_records, f, ensure_ascii=False, indent=2)

print(f"Saved: {OUTPUT_JSON}")
print(f"   {len(output_records)} brand records total")